In [1]:
import cv2
import os
import numpy as np
from pathlib import Path

In [ ]:
# ----- Parameters ------
input_image_path = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/global_images/15.jpg'
input_mask_path = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/global_images/15_mask_inv.jpg'
# input_mask_path = ''
output_tiles_path = r'/home/tinmar/Desktop/Projects/Datasets/Puzzle/tiles/'

is_mask_inv = 'inv' in Path(input_mask_path).stem
if is_mask_inv:
    print("Using inverted mask logic.")

image_number = int(Path(input_image_path).stem)
if str(image_number) not in Path(input_mask_path).stem:
    raise ValueError("Image number and mask number do not match.")

if not os.path.exists(input_image_path):
    raise ValueError(f"Image path does not exist: {input_image_path}")
if input_mask_path != '' and not os.path.exists(input_mask_path):
    print(f"Edges rough image path does not exist: {input_mask_path}")


Using inverted mask logic.
Edges rough image path does not exist: /home/tinmar/Desktop/Projects/Datasets/Puzzle/global_images/15_mask_inv.png


In [3]:
def create_tiles(image_path, mask_path, output_dir, tile_size=512, overlap=32):

    if input_mask_path == '':
        mask_provided = False
        print("No mask path provided, tiling only the image.")
    else:
        mask_provided = True


    # Load images
    img = cv2.imread(image_path)
    if mask_provided:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    im_name = Path(image_path).stem
    
    h, w = img.shape[:2]
    stride = tile_size - overlap
    
    # Create output directories
    os.makedirs(f"{output_dir}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/masks", exist_ok=True)

    count = 0
    for y in range(0, h, stride):
        for x in range(0, w, stride):
            # Adjust coordinates if tile goes out of bounds
            y_start = y
            x_start = x
            
            if y_start + tile_size > h:
                y_start = h - tile_size
            if x_start + tile_size > w:
                x_start = w - tile_size
                
            y_end = y_start + tile_size
            x_end = x_start + tile_size

            # Extract sub-images
            img_tile = img[y_start:y_end, x_start:x_end]
            if mask_provided:  
                mask_tile = mask[y_start:y_end, x_start:x_end]

            # Save tiles
            cv2.imwrite(f"{output_dir}/images/{im_name}_{count}.png", img_tile)
            if mask_provided:
                if is_mask_inv:
                    cv2.imwrite(f"{output_dir}/masks_inv/{im_name}_{count}.png", mask_tile)
                else:
                    cv2.imwrite(f"{output_dir}/masks/{im_name}_{count}.png", mask_tile)
                    
            
            count += 1
            
            # Break if we've reached the edge
            if x_start + tile_size >= w:
                break
        if y_start + tile_size >= h:
            break

    print(f"Created {count} sub-image pairs.")

In [4]:
create_tiles(input_image_path, input_mask_path, output_tiles_path, tile_size=640, overlap=10)

[ WARN:0@0.050] global loadsave.cpp:275 findDecoder imread_('/home/tinmar/Desktop/Projects/Datasets/Puzzle/global_images/15_mask_inv.png'): can't open/read file: check file path/integrity


TypeError: 'NoneType' object is not subscriptable